In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   VideoGuard — Audio Moderation Service (Colab + ngrok)                ║
# ║   faster-whisper + PhoBERT                                                ║
# ║   Runtime: GPU T4 recommended  |  Port: 8000                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
WHISPER_MODEL_SIZE     = "large-v3"   # tiny|base|small|medium|large-v2|large-v3|distil-large-v3
# WHISPER_MODEL_SIZE   = "medium"     # nhẹ hơn nếu thiếu VRAM
WHISPER_LANGUAGE       = "vi"
WHISPER_INITIAL_PROMPT = ""  # để trống; prompt dài dễ hallucination
WHISPER_BEAM_SIZE      = 5
PHOBERT_MODEL_PATH     = "/content/drive/MyDrive/toxic_clean_model"
PHOBERT_BASE_MODEL     = "vinai/phobert-base-v2"
PHOBERT_NUM_LABELS     = 2
PHOBERT_DROPOUT        = 0.2
PHOBERT_UNFREEZE_LAST_N = 8
NGROK_AUTHTOKEN        = ""
PORT                   = 8000
PHOBERT_MAX_LENGTH     = 128
PHOBERT_BATCH_SIZE     = 32
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

# ── Bước 1: Kiểm tra và fix numpy trước tiên ─────────────────────────────────
# pyvi==0.1.1 dùng pickle với numpy cũ → không tương thích numpy>=2.0
# Phải downgrade numpy==1.26.4 và restart kernel
import numpy as _np_check
_np_major = int(_np_check.__version__.split(".")[0])
if _np_major >= 2:
    print(f"⚠️  Phát hiện numpy {_np_check.__version__} (không tương thích với pyvi).")
    print("🔧 Đang downgrade numpy==1.26.4 và restart kernel...")
    _pip("--force-reinstall", "numpy==1.26.4")
    print("✅ numpy đã được cài lại. Kernel sẽ tự restart sau 2 giây...")
    import time; time.sleep(2)
    os.kill(os.getpid(), 9)   # force-restart kernel — cell sẽ tự chạy lại

print(f"✅ numpy {_np_check.__version__} — OK, tiếp tục cài đặt...")

# ── Bước 2: Cài các thư viện còn lại ─────────────────────────────────────────
print("📦 Đang cài đặt thư viện...")
_pip(
    "fastapi==0.115.5",
    "uvicorn[standard]==0.32.1",
    "python-multipart==0.0.18",
    "faster-whisper==1.1.1",
    "transformers==4.46.3",
    "pyvi==0.1.1",
    "pyngrok",
    "nest_asyncio",
)
print("✅ Cài đặt xong!")

# ── Bước 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Bước 4: Imports ───────────────────────────────────────────────────────────
import logging
import tempfile
import threading

import numpy as np
import torch
import torch.nn as nn
import nest_asyncio
nest_asyncio.apply()

from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModel
from pyvi import ViTokenizer
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
)
logger = logging.getLogger("audio_moderation")

# ── Bước 5: Pydantic schemas ──────────────────────────────────────────────────
class SentencePrediction(BaseModel):
    text:       str
    label:      str
    label_id:   int
    confidence: float
    scores:     dict
    start_sec:  float
    end_sec:    float

class AudioPredictResponse(BaseModel):
    total_sentences: int
    flagged_count:   int
    overall_label:   str
    sentences:       list

class HealthResponse(BaseModel):
    status:         str
    whisper_loaded: bool
    phobert_loaded: bool
    device:         str

# ── Bước 6: Constants ─────────────────────────────────────────────────────────
# Model được train theo binary classification: 0=Clean, 1=Toxic
LABEL_MAP      = {0: "Clean", 1: "Toxic"}
FLAGGED_LABELS = {"Toxic"}
LABEL_PRIORITY = {"Toxic": 1, "Clean": 0}

# ── Bước 7: CustomPhoBERT Architecture ───────────────────────────────────────
# PHẢI khớp chính xác với architecture được dùng khi training trên Kaggle.
# CLS + Mean Pooling → 2-layer GELU classifier head.
class CustomPhoBERT(nn.Module):
    """
    PhoBERT fine-tuning với 3 cải tiến:
      1. Partial unfreeze: N transformer layers cuối được unfreeze để fine-tune.
      2. CLS + Mean Pooling: kết hợp [CLS] token và mean pooling → richer representation.
      3. 2-layer classifier head với GELU activation.
    """
    def __init__(self, model_name, num_labels, dropout_prob, unfreeze_last_n=4):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.config  = self.phobert.config
        hidden_size  = self.config.hidden_size

        # Freeze toàn bộ backbone
        for param in self.phobert.parameters():
            param.requires_grad = False

        # Unfreeze N transformer layers cuối
        encoder_layers = self.phobert.encoder.layer
        for layer in encoder_layers[-unfreeze_last_n:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Unfreeze pooler nếu có
        if hasattr(self.phobert, "pooler") and self.phobert.pooler is not None:
            for param in self.phobert.pooler.parameters():
                param.requires_grad = True

        # Classifier head 2-layer với GELU
        # Input = CLS vector + Mean Pooling vector → hidden_size * 2
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size // 2, num_labels),
        )

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )
        last_hidden_state = outputs.last_hidden_state  # (B, T, H)

        # CLS token embedding
        cls_output = last_hidden_state[:, 0, :]         # (B, H)

        # Mean Pooling (masked)
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, dim=1)
        sum_mask       = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled    = sum_embeddings / sum_mask       # (B, H)

        # Concatenate CLS + Mean → (B, H*2) → classifier
        pooled = torch.cat([cls_output, mean_pooled], dim=-1)
        logits = self.classifier(pooled)

        class Output:
            def __init__(self, logits):
                self.logits = logits
        return Output(logits)

    @classmethod
    def from_pretrained(cls, save_directory, base_model_name, num_labels, dropout_prob, unfreeze_last_n=4):
        model = cls(base_model_name, num_labels, dropout_prob, unfreeze_last_n)
        state_dict_path = os.path.join(save_directory, "pytorch_model.bin")
        if not os.path.exists(state_dict_path):
          raise FileNotFoundError(f"❌ Không tìm thấy file trọng số: {state_dict_path}")

        try:
            # PyTorch 2.4+ mặc định weights_only=True, có thể gây lỗi. Ta set False.
            state_dict = torch.load(state_dict_path, map_location="cpu", weights_only=False)
            model.load_state_dict(state_dict)
        except Exception as e:
            if "failed reading zip archive" in str(e):
                raise RuntimeError(
                    f"\n{'='*60}\n"
                    f"❌ LỖI NGHIÊM TRỌNG: File {state_dict_path} bị hỏng (corrupted)!\n"
                    "Nguyên nhân: Quá trình upload file pytorch_model.bin (447MB) lên Google Drive bị lỗi hoặc chưa hoàn tất.\n"
                    "👉 CÁCH KHẮC PHỤC:\n"
                    "  1. Xóa file pytorch_model.bin hiện tại trên Google Drive.\n"
                    "  2. Upload lại file pytorch_model.bin mới từ máy tính (chờ cho đến khi upload xong 100%).\n"
                    "  3. Chạy lại cell này.\n"
                    f"{'='*60}\n"
                ) from e
            raise e

        return model

# ── Bước 8: Model singletons ──────────────────────────────────────────────────
_whisper   = None
_tokenizer = None
_model     = None
_device    = None

def load_whisper():
    global _whisper
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8_float16"
    logger.info(f"🎙️  Loading faster-whisper {WHISPER_MODEL_SIZE} on {device} ({compute_type})...")
    try:
        _whisper = WhisperModel(WHISPER_MODEL_SIZE, device=device, compute_type=compute_type, cpu_threads=4)
    except ValueError:
        _whisper = WhisperModel(WHISPER_MODEL_SIZE, device=device, compute_type="int8", cpu_threads=4)
    logger.info("✅ faster-whisper loaded")

def load_phobert():
    global _tokenizer, _model, _device
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"🧠 Loading CustomPhoBERT from: {PHOBERT_MODEL_PATH}  (device={_device})")

    # Tokenizer được lưu cùng thư mục model (tokenizer.save_pretrained)
    _tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL_PATH)

    # Load CustomPhoBERT với đúng architecture đã dùng khi training
    _model = CustomPhoBERT.from_pretrained(
        save_directory   = PHOBERT_MODEL_PATH,
        base_model_name  = PHOBERT_BASE_MODEL,
        num_labels       = PHOBERT_NUM_LABELS,
        dropout_prob     = PHOBERT_DROPOUT,
        unfreeze_last_n  = PHOBERT_UNFREEZE_LAST_N,
    ).to(_device)
    _model.eval()
    logger.info(f"✅ CustomPhoBERT loaded (device={_device}, labels={PHOBERT_NUM_LABELS})")

# ── Bước 9: Core logic (faster-whisper) ───────────────────────────────────────
import re
import unicodedata

_HALLUCINATION_FRAGMENTS = (
    "subscribe", "đăng ký", "đăng kí", "kênh youtube", "ghiền mì gõ", "ghien mi go",
    "không bỏ lỡ", "video hấp dẫn", "để không bỏ lỡ những video", "cảm ơn các bạn đã theo dõi",
    "hãy like", "bấm chuông", "theo dõi kênh", "xem video tiếp theo",
)
def _norm_text(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", t.strip().lower()))

def _is_spam_segment(text):
    n = _norm_text(text)
    if len(n) < 3:
        return True
    return any(f in n for f in _HALLUCINATION_FRAGMENTS)

def _whisper_once(audio_path, use_vad):
    kw = {
        "language": WHISPER_LANGUAGE,
        "word_timestamps": True,
        "vad_filter": use_vad,
        "condition_on_previous_text": False,
        "temperature": 0.0,
        "compression_ratio_threshold": 2.4,
        "no_speech_threshold": 0.5,
        "beam_size": WHISPER_BEAM_SIZE,
        "best_of": WHISPER_BEAM_SIZE,
    }
    if use_vad:
        kw["vad_parameters"] = {"min_silence_duration_ms": 500, "speech_pad_ms": 300}
    if WHISPER_INITIAL_PROMPT.strip():
        kw["initial_prompt"] = WHISPER_INITIAL_PROMPT.strip()
    segs, info = _whisper.transcribe(audio_path, **kw)
    return list(segs), info

def transcribe_audio(audio_path: str):
    """Segment Whisper gốc; upload từ server: mono 16 kHz WAV."""
    raw, info = _whisper_once(audio_path, use_vad=True)
    sentences = []
    for seg in raw:
        text = (seg.text or "").strip()
        if not text or _is_spam_segment(text):
            if text:
                logger.info("Dropped spam: %s", text[:80])
            continue
        start, end = float(seg.start), float(seg.end)
        if end <= start:
            end = start + 0.1
        sentences.append({"text": text, "start_sec": start, "end_sec": end})
    if not sentences:
        logger.warning("0 segments — retry without VAD")
        raw, info = _whisper_once(audio_path, False)
        for seg in raw:
            text = (seg.text or "").strip()
            if not text or _is_spam_segment(text):
                continue
            start, end = float(seg.start), float(seg.end)
            if end <= start:
                end = start + 0.1
            sentences.append({"text": text, "start_sec": start, "end_sec": end})
    if sentences:
        logger.info("Sample timings: %s", [(round(s["start_sec"], 2), round(s["end_sec"], 2), s["text"][:50]) for s in sentences[:5]])
    logger.info(f"📝 lang={info.language} prob={info.language_probability:.2f} raw={len(raw)} kept={len(sentences)}")
    return sentences

@torch.no_grad()
def phobert_predict(texts):
    """
    Dự đoán nhãn (Clean / Toxic) cho danh sách văn bản.
    Áp dụng word segmentation nhất quán với quá trình training.
    """
    # QUAN TRỌNG: word-segment giống hệt bước training
    segmented = [ViTokenizer.tokenize(str(t).strip().lower()) for t in texts]
    results = []
    for i in range(0, len(segmented), PHOBERT_BATCH_SIZE):
        batch   = segmented[i : i + PHOBERT_BATCH_SIZE]
        encoded = _tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=PHOBERT_MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(_device) for k, v in encoded.items()}
        # CustomPhoBERT.forward trả về object có thuộc tính .logits
        probs   = torch.softmax(_model(**encoded).logits, dim=-1).cpu().numpy()
        for prob in probs:
            pid = int(np.argmax(prob))
            results.append({
                "label":      LABEL_MAP[pid],
                "label_id":   pid,
                "confidence": float(prob[pid]),
                "scores":     {
                    "Clean": float(prob[0]),
                    "Toxic": float(prob[1]),
                },
            })
    return results

def worst_label(sentences):
    worst = "Clean"
    for s in sentences:
        if LABEL_PRIORITY.get(s.label, 0) > LABEL_PRIORITY.get(worst, 0):
            worst = s.label
    return worst

# ── Bước 10: Load models ───────────────────────────────────────────────────────
print("\n🚀 Đang load models...")
load_whisper()
load_phobert()
print("✅ Tất cả models đã sẵn sàng!\n")

# ── Bước 11: FastAPI app ──────────────────────────────────────────────────────
app = FastAPI(
    title="VideoGuard Audio Moderation",
    description="faster-whisper + PhoBERT (Vietnamese audio moderation).",
    version="2.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

@app.get("/health", response_model=HealthResponse, tags=["health"])
def health():
    return HealthResponse(
        status="ok",
        whisper_loaded=_whisper is not None,
        phobert_loaded=_model is not None,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

@app.post("/audio/predict", response_model=AudioPredictResponse, tags=["predict"])
async def predict_audio(file: UploadFile = File(..., description="Audio file WAV/MP3")):
    suffix = os.path.splitext(file.filename or "audio.wav")[1] or ".wav"
    try:
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp.write(await file.read())
            tmp_path = tmp.name
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Could not save audio: {exc}")

    try:
        segments = transcribe_audio(tmp_path)
        if not segments:
            return AudioPredictResponse(
                total_sentences=0, flagged_count=0,
                overall_label="Clean", sentences=[]
            )
        texts   = [s["text"] for s in segments]
        preds   = phobert_predict(texts)
        results = [
            SentencePrediction(
                text=seg["text"],
                start_sec=seg["start_sec"],
                end_sec=seg["end_sec"],
                **pred,
            )
            for seg, pred in zip(segments, preds)
        ]
        flagged = sum(1 for s in results if s.label in FLAGGED_LABELS)
        verdict = worst_label(results)
        logger.info(f"[predict] sentences={len(results)} flagged={flagged} verdict={verdict}")
        return AudioPredictResponse(
            total_sentences=len(results),
            flagged_count=flagged,
            overall_label=verdict,
            sentences=results,
        )
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass

# ── Bước 12: Khởi động ngrok + uvicorn ───────────────────────────────────────
from pyngrok import ngrok, conf

if not NGROK_AUTHTOKEN:
    raise ValueError("❌ Hãy điền NGROK_AUTHTOKEN ở phần CẤU HÌNH phía trên!")

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()

public_url = ngrok.connect(PORT, "http").public_url
print(f"\n{'='*60}")
print(f"🌐 Public API URL : {public_url}")
print(f"📖 Swagger Docs   : {public_url}/docs")
print(f"❤️  Health Check  : {public_url}/health")
print(f"🔊 Predict Audio  : POST {public_url}/audio/predict")
print(f"{'='*60}\n")

# Chạy uvicorn (blocking — cell này chạy liên tục cho đến khi bạn dừng)
uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
